# Breakout 4B · Sprint 3 · what each tax number serves

From the customs registers: the importer codes clearing under a number, its consignees, the addresses and brokers behind them. Then the rule, and what the tax register says. **T103629 is worked.**

The same steps as the sheet S3 Network of `4B_compute.xlsx` (built from the raw rows of the extract). Stuck? `4B_compute_worked.xlsx` shows every formula.

*Published customs datasets record an irregularity on 1 to 8 lines in 100; in this lab the random draw finds 3 in 100.*

In [ ]:
# Run this cell first (click it, then Shift + Enter). It finds the lab data.
import hashlib, os
from pathlib import Path
import numpy as np, pandas as pd

CANDIDATES = [os.environ.get('WORKSHOP_DATA', ''), 'data', '../data', '/workspace/data', '.']
DATA = next(Path(c) for c in CANDIDATES if c and (Path(c) / 'tariff.csv').exists())
print('Data folder:', DATA)

def _h(v, nd):
    return hashlib.sha256(f'{round(float(v), nd):.{nd}f}'.encode()).hexdigest()[:16]
def _hs(v):
    return hashlib.sha256(str(v).strip().lower().encode()).hexdigest()[:16]
def check(name, value, expected, nd=None):
    if callable(value):
        try:
            value = value()
        except Exception:
            value = ...
    try:
        ok = value is not ... and (_hs(value) == expected if nd is None else _h(value, nd) == expected)
    except Exception:
        ok = False
    shown = value if nd is None or value is ... else f'{value:,.{nd}f}'
    print(('✓ ' if ok else '✗ not yet: ') + name + ' = ' + str(shown))


In [ ]:
importers = pd.read_csv(DATA / 'importers.csv')
consignees = pd.read_csv(DATA / 'consignees.csv')
register = pd.read_csv(DATA / 'tax_registry.csv')
imp25 = pd.read_csv(DATA / 'importers_4b.csv')

## Worked: T103629

In [ ]:
t, own = 'T103629', 'IMP00519'
g = importers[importers.tin == t]
names = len(g) - 1                                          # other importer codes clearing under the number
cons = names + (consignees.importer_id == own).sum()      # every party clearing under it
print('names', names, '· consignees', cons, '· addresses', g.address_line.nunique(), '· brokers', g.default_broker_id.nunique())

## YOUR TURN: every tax number of the 25

In [ ]:
# YOUR TURN: one row per tax number of the 25, as for T103629 above
numbers = imp25.dropna(subset=['tin_customs'])[['tin_customs', 'importer_id']]   # the number, and the importer's own code
counts = ...   # index: the tax number; columns consignees, addresses, director_companies (how many companies its director runs in the register)

In [ ]:
E = {'T100033': {'consignees': '4fc82b26aecb47d2', 'addresses': 'd4735e3a265e16ee', 'director_companies': '7902699be42c8a8e'}, 'T100070': {'consignees': 'e629fa6598d73276', 'addresses': '6b86b273ff34fce1', 'director_companies': 'e7f6c011776e8db7'}, 'T100440': {'consignees': '6b86b273ff34fce1', 'addresses': '6b86b273ff34fce1', 'director_companies': 'ef2d127de37b942b'}, 'T100466': {'consignees': '4b227777d4dd1fc6', 'addresses': '6b86b273ff34fce1', 'director_companies': 'd4735e3a265e16ee'}, 'T102011': {'consignees': '4e07408562bedb8b', 'addresses': '6b86b273ff34fce1', 'director_companies': None}, 'T102141': {'consignees': 'd4735e3a265e16ee', 'addresses': '6b86b273ff34fce1', 'director_companies': '4b227777d4dd1fc6'}, 'T102376': {'consignees': '6b86b273ff34fce1', 'addresses': '6b86b273ff34fce1', 'director_companies': '4b227777d4dd1fc6'}, 'T103130': {'consignees': '6b86b273ff34fce1', 'addresses': '6b86b273ff34fce1', 'director_companies': '4e07408562bedb8b'}, 'T103425': {'consignees': '6b86b273ff34fce1', 'addresses': '6b86b273ff34fce1', 'director_companies': 'ef2d127de37b942b'}, 'T103629': {'consignees': 'd4735e3a265e16ee', 'addresses': '6b86b273ff34fce1', 'director_companies': 'ef2d127de37b942b'}, 'T104498': {'consignees': '4e07408562bedb8b', 'addresses': '6b86b273ff34fce1', 'director_companies': '6b51d431df5d7f14'}, 'T105264': {'consignees': '6b86b273ff34fce1', 'addresses': '6b86b273ff34fce1', 'director_companies': '4e07408562bedb8b'}, 'T106240': {'consignees': '4e07408562bedb8b', 'addresses': '6b86b273ff34fce1', 'director_companies': 'e629fa6598d73276'}, 'T106477': {'consignees': 'd4735e3a265e16ee', 'addresses': '6b86b273ff34fce1', 'director_companies': '4b227777d4dd1fc6'}, 'T106987': {'consignees': 'd4735e3a265e16ee', 'addresses': '6b86b273ff34fce1', 'director_companies': 'd4735e3a265e16ee'}, 'T107258': {'consignees': '6b86b273ff34fce1', 'addresses': '6b86b273ff34fce1', 'director_companies': 'ef2d127de37b942b'}, 'T107617': {'consignees': '4e07408562bedb8b', 'addresses': '6b86b273ff34fce1', 'director_companies': 'd4735e3a265e16ee'}, 'T107992': {'consignees': '6b86b273ff34fce1', 'addresses': '6b86b273ff34fce1', 'director_companies': '6b86b273ff34fce1'}, 'T108692': {'consignees': '6b86b273ff34fce1', 'addresses': '6b86b273ff34fce1', 'director_companies': '4e07408562bedb8b'}, 'T109167': {'consignees': '4e07408562bedb8b', 'addresses': '6b86b273ff34fce1', 'director_companies': '4e07408562bedb8b'}}
def right(column):   # how many numbers carry the right value: the expected values are hashed, none is printed
    return sum(_h(counts.loc[t, column], 0) == e[column] for t, e in E.items() if e[column])
check('numbers with the right consignees (of 20)', lambda: right('consignees'), 'f5ca38f748a1d6ea', 0)
check('numbers with the right addresses (of 20)', lambda: right('addresses'), 'f5ca38f748a1d6ea', 0)
check('numbers with the right companies of their director (of 19)', lambda: right('director_companies'), '9400f1b21cb527d7', 0)

The rule: 8 or more consignees, one address, sales under 20% of imports. Which number does it find, and which only the register shows? Treat the cluster, not the single importer, as the case.